<a href="https://colab.research.google.com/github/mustaphaismail225-beep/prediction-taux-interet/blob/main/phase1_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prédiction du taux d'intérêt d'un prêt bancaire
## Phase 1 : Collecte et exploration des données

**Auteur :** Mustapha Ismail – Bootcamp Data Scientist GoMyCode
**Objectif :** prédire le taux d'intérêt d'un prêt à partir du profil de l'emprunteur (régression).
**Données :** prêts Lending Club (Kaggle), 396 030 prêts et 27 colonnes.

### Démarche
1. **Chargement** : les données sont stockées sur Google Drive et chargées avec pandas.
2. **Vue d'ensemble** : un tableau résume le type, les valeurs manquantes et la description de chaque colonne.
3. **Choix des variables** : je supprime les variables calculées à partir du taux (fuite de données), le statut final du prêt et les textes libres.
4. **Conversion des types** : la durée et l'ancienneté deviennent des nombres, et les dates sont converties au format date.
5. **Valeurs manquantes** : je supprime les lignes très rares, et l'imputation par la médiane sera faite dans le pipeline pour éviter une fuite de données.
6. **Valeurs aberrantes** : je supprime seulement les valeurs impossibles (revenu nul, endettement > 100 %, utilisation du crédit > 150 %).
7. **Feature engineering** : je crée l'année du prêt, l'ancienneté du crédit, le ratio prêt / revenu et le logarithme des montants extrêmes.
8. **Fuite cachée** : le code postal révèle le statut final du prêt (100 % remboursés ou 100 % en défaut), donc je le supprime.
9. **Encodage** : je regroupe les catégories rares et j'encode les variables texte en colonnes 0/1.
10. **Sauvegarde** : 395 207 prêts conservés, en version lisible (20 colonnes) et en version encodée pour le modèle (33 colonnes).

In [16]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

chemin = "/content/drive/MyDrive/Projet_Taux_Interet/data/"
df = pd.read_csv(chemin + "lending_club_loan_two.csv")
info = pd.read_csv(chemin + "lending_club_info.csv")

print("Dimensions :", df.shape)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dimensions : (396030, 27)


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,title,dti,earliest_cr_line,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies,address
0,10000.0,36 months,11.44,329.48,B,B4,Marketing,10+ years,RENT,117000.0,Not Verified,Jan-2015,Fully Paid,vacation,Vacation,26.24,Jun-1990,16.0,0.0,36369.0,41.8,25.0,w,INDIVIDUAL,0.0,0.0,"0174 Michelle Gateway\nMendozaberg, OK 22690"
1,8000.0,36 months,11.99,265.68,B,B5,Credit analyst,4 years,MORTGAGE,65000.0,Not Verified,Jan-2015,Fully Paid,debt_consolidation,Debt consolidation,22.05,Jul-2004,17.0,0.0,20131.0,53.3,27.0,f,INDIVIDUAL,3.0,0.0,"1076 Carney Fort Apt. 347\nLoganmouth, SD 05113"
2,15600.0,36 months,10.49,506.97,B,B3,Statistician,< 1 year,RENT,43057.0,Source Verified,Jan-2015,Fully Paid,credit_card,Credit card refinancing,12.79,Aug-2007,13.0,0.0,11987.0,92.2,26.0,f,INDIVIDUAL,0.0,0.0,"87025 Mark Dale Apt. 269\nNew Sabrina, WV 05113"
3,7200.0,36 months,6.49,220.65,A,A2,Client Advocate,6 years,RENT,54000.0,Not Verified,Nov-2014,Fully Paid,credit_card,Credit card refinancing,2.60,Sep-2006,6.0,0.0,5472.0,21.5,13.0,f,INDIVIDUAL,0.0,0.0,"823 Reid Ford\nDelacruzside, MA 00813"
4,24375.0,60 months,17.27,609.33,C,C5,Destiny Management Inc.,9 years,MORTGAGE,55000.0,Verified,Apr-2013,Charged Off,credit_card,Credit Card Refinance,33.95,Mar-1999,13.0,0.0,24584.0,69.8,43.0,f,INDIVIDUAL,1.0,0.0,"679 Luna Roads\nGreggshire, VA 11650"


In [17]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

resume = pd.DataFrame({
    "type": df.dtypes,
    "manquants": df.isnull().sum(),
    "% manquants": (df.isnull().mean() * 100).round(2),
    "valeurs uniques": df.nunique()
})

resume = resume.merge(info.set_index("LoanStatNew"),
                      left_index=True, right_index=True, how="left")

print("Doublons :", df.duplicated().sum())
resume

Doublons : 0


,type,manquants,% manquants,valeurs uniques,Description
loan_amnt,float64,0,0.00,1397,The listed amount of the loan applied for by the borrower. If at some point ...
term,object,0,0.00,2,The number of payments on the loan. Values are in months and can be either 3...
int_rate,float64,0,0.00,566,Interest Rate on the loan
installment,float64,0,0.00,55706,The monthly payment owed by the borrower if the loan originates.
grade,object,0,0.00,7,LC assigned loan grade
sub_grade,object,0,0.00,35,LC assigned loan subgrade
emp_title,object,22927,5.79,173105,The job title supplied by the Borrower when applying for the loan.*
emp_length,object,18301,4.62,11,Employment length in years. Possible values are between 0 and 10 where 0 mea...
home_ownership,object,0,0.00,6,The home ownership status provided by the borrower during registration or ob...
annual_inc,float64,0,0.00,27197,The self-reported annual income provided by the borrower during registration.


In [18]:
cible = "int_rate"

a_supprimer = ["grade", "sub_grade", "installment",
               "loan_status", "emp_title", "title"]

df_clean = df.drop(columns=a_supprimer).copy()

print("Avant :", df.shape)
print("Après :", df_clean.shape)
print(df_clean.columns.tolist())

Avant : (396030, 27)
Après : (396030, 21)
['loan_amnt', 'term', 'int_rate', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'purpose', 'dti', 'earliest_cr_line', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'application_type', 'mort_acc', 'pub_rec_bankruptcies', 'address']


In [19]:
cols = ["term", "emp_length", "issue_d", "earliest_cr_line"]
print("AVANT :")
print(df_clean[cols].head(3))

# 1) term : "36 months" -> 36
df_clean["term"] = df_clean["term"].str.extract(r"(\d+)")[0].astype(int)

# 2) emp_length : "< 1 year" -> 0, "10+ years" -> 10
df_clean["emp_length"] = (df_clean["emp_length"]
                          .replace("< 1 year", "0 years")
                          .str.extract(r"(\d+)")[0]
                          .astype(float))

# 3) Dates : "Jan-2015" -> date
df_clean["issue_d"] = pd.to_datetime(df_clean["issue_d"], format="%b-%Y", errors="coerce")
df_clean["earliest_cr_line"] = pd.to_datetime(df_clean["earliest_cr_line"], format="%b-%Y", errors="coerce")

print("\nAPRÈS :")
print(df_clean[cols].head(3))
print("\nTypes :")
print(df_clean[cols].dtypes)
print("\nManquants :")
print(df_clean[cols].isnull().sum())

AVANT :
         term emp_length   issue_d earliest_cr_line
0   36 months  10+ years  Jan-2015         Jun-1990
1   36 months    4 years  Jan-2015         Jul-2004
2   36 months   < 1 year  Jan-2015         Aug-2007

APRÈS :
   term  emp_length    issue_d earliest_cr_line
0    36        10.0 2015-01-01       1990-06-01
1    36         4.0 2015-01-01       2004-07-01
2    36         0.0 2015-01-01       2007-08-01

Types :
term                         int64
emp_length                 float64
issue_d             datetime64[ns]
earliest_cr_line    datetime64[ns]
dtype: object

Manquants :
term                    0
emp_length          18301
issue_d                 0
earliest_cr_line        0
dtype: int64


In [20]:
avant = len(df_clean)

df_clean = df_clean.dropna(subset=["revol_util", "pub_rec_bankruptcies"])

print("Lignes supprimées :", avant - len(df_clean))
print("Lignes restantes :", len(df_clean))
print("\nManquants restants :")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print("\nMédianes (pour information) :")
print(df_clean[["emp_length", "mort_acc"]].median())

Lignes supprimées : 811
Lignes restantes : 395219

Manquants restants :
emp_length    18290
mort_acc      37205
dtype: int64

Médianes (pour information) :
emp_length    6.0
mort_acc      1.0
dtype: float64


In [21]:
num = df_clean.select_dtypes("number")

stats = num.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T

q1 = num.quantile(0.25)
q3 = num.quantile(0.75)
iqr = q3 - q1
outliers = ((num < q1 - 1.5 * iqr) | (num > q3 + 1.5 * iqr)).sum()

stats["nb_outliers_IQR"] = outliers
stats["% outliers"] = (outliers / len(num) * 100).round(2)

stats[["min", "1%", "50%", "99%", "max", "nb_outliers_IQR", "% outliers"]].round(2)

,min,1%,50%,99%,max,nb_outliers_IQR,% outliers
loan_amnt,500.00,1600.00,12000.00,35000.00,40000.00,190,0.05
term,36.00,36.00,36.00,60.00,60.00,93972,23.78
int_rate,5.32,6.00,13.33,25.28,30.99,3144,0.80
emp_length,0.00,0.00,6.00,10.00,10.00,0,0.00
annual_inc,0.00,19000.00,64000.00,250000.00,8706582.00,16649,4.21
dti,0.00,1.58,16.92,36.44,9999.00,275,0.07
open_acc,1.00,3.00,10.00,27.00,90.00,10297,2.61
pub_rec,0.00,0.00,0.00,2.00,86.00,57730,14.61
revol_bal,0.00,184.00,11190.00,85856.48,1743266.00,21205,5.37
revol_util,0.00,1.10,54.80,98.00,892.30,12,0.00


In [22]:
avant = len(df_clean)

cond_inc = df_clean["annual_inc"] > 0
cond_dti = df_clean["dti"] <= 100
cond_rev = df_clean["revol_util"] <= 150

print("Revenu = 0 :", (~cond_inc).sum())
print("DTI > 100 :", (~cond_dti).sum())
print("Revol_util > 150 :", (~cond_rev).sum())

df_clean = df_clean[cond_inc & cond_dti & cond_rev]

print("\nLignes supprimées :", avant - len(df_clean))
print("Lignes restantes :", len(df_clean))
print(df_clean[["annual_inc", "dti", "revol_util"]].describe().round(2))

Revenu = 0 : 1
DTI > 100 : 8
Revol_util > 150 : 4

Lignes supprimées : 12
Lignes restantes : 395207
       annual_inc        dti  revol_util
count   395207.00  395207.00   395207.00
mean     74200.63      17.36       53.81
std      61557.20       8.13       24.40
min       4000.00       0.00        0.00
25%      45000.00      11.30       35.90
50%      64000.00      16.92       54.80
75%      90000.00      22.99       72.90
max    8706582.00      93.86      148.00


In [23]:
df_clean.to_parquet(chemin + "df_clean_etape4.parquet", index=False)
print("Sauvegardé ✅", df_clean.shape)

Sauvegardé ✅ (395207, 21)


In [24]:
df_fe = df_clean.copy()

# 1) À partir des dates
df_fe["annee_emission"] = df_fe["issue_d"].dt.year
df_fe["anciennete_credit"] = ((df_fe["issue_d"] - df_fe["earliest_cr_line"]).dt.days / 365.25).round(1)

# 2) À partir de l'adresse
df_fe["code_postal"] = df_fe["address"].str[-5:]
df_fe["etat"] = df_fe["address"].str[-8:-6]

# Vérifications
print(df_fe[["address", "etat", "code_postal"]].head(3))
print("\nNombre d'états :", df_fe["etat"].nunique())
print("Nombre de codes postaux :", df_fe["code_postal"].nunique())
print("\nAnnées :", sorted(df_fe["annee_emission"].unique()))
print("\nAncienneté du crédit (ans) :")
print(df_fe["anciennete_credit"].describe().round(1))

                                           address etat code_postal
0     0174 Michelle Gateway\nMendozaberg, OK 22690   OK       22690
1  1076 Carney Fort Apt. 347\nLoganmouth, SD 05113   SD       05113
2  87025 Mark Dale Apt. 269\nNew Sabrina, WV 05113   WV       05113

Nombre d'états : 54
Nombre de codes postaux : 10

Années : [np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016)]

Ancienneté du crédit (ans) :
count    395207.0
mean         15.8
std           7.2
min           3.0
25%          10.9
50%          14.3
75%          19.3
max          70.2
Name: anciennete_credit, dtype: float64


In [25]:
print("Taux moyen général :", round(df_fe["int_rate"].mean(), 2))

print("\nTaux moyen par code postal :")
print(df_fe.groupby("code_postal")["int_rate"].agg(["mean", "count"]).round(2))

moy_etat = df_fe.groupby("etat")["int_rate"].mean()
print("\nTaux moyen par état : min =", round(moy_etat.min(), 2),
      "| max =", round(moy_etat.max(), 2))

Taux moyen général : 13.64

Taux moyen par code postal :
              mean  count
code_postal              
00813        13.08  45723
05113        13.08  45297
11650        15.91  11210
22690        13.63  56412
29597        13.12  45393
30723        13.65  56400
48052        13.65  55809
70466        13.65  56878
86630        15.91  10959
93700        15.83  11126

Taux moyen par état : min = 13.52 | max = 13.75


In [26]:
verif = pd.crosstab(df["address"].str[-5:], df["loan_status"], normalize="index")
(verif * 100).round(1)

loan_status,Charged Off,Fully Paid
address,,
00813,0.0,100.0
05113,0.0,100.0
11650,100.0,0.0
22690,19.4,80.6
29597,0.0,100.0
30723,19.5,80.5
48052,20.0,80.0
70466,19.6,80.4
86630,100.0,0.0


Les adresses du jeu de données sont fictives : il n'y a que 10 codes postaux, et le même code apparaît dans plusieurs États. L'État n'a aucun lien avec le taux. Le code postal semblait utile, mais un tableau croisé montre qu'il révèle directement le statut final du prêt : certains codes sont 100 % remboursés, d'autres 100 % en défaut. C'est une fuite de données cachée, donc j'ai supprimé ces variables.

In [27]:
import numpy as np

# 1) Supprimer les colonnes inutiles
df_fe = df_fe.drop(columns=["address", "etat", "code_postal",
                            "issue_d", "earliest_cr_line"])

# 2) Variable métier (avant le log, avec les vraies valeurs)
df_fe["ratio_pret_revenu"] = (df_fe["loan_amnt"] / df_fe["annual_inc"]).round(3)

# 3) Transformation log
print("Asymétrie AVANT :")
print(df_fe[["annual_inc", "revol_bal"]].skew().round(2))

df_fe["log_annual_inc"] = np.log1p(df_fe["annual_inc"])
df_fe["log_revol_bal"] = np.log1p(df_fe["revol_bal"])
df_fe = df_fe.drop(columns=["annual_inc", "revol_bal"])

print("\nAsymétrie APRÈS :")
print(df_fe[["log_annual_inc", "log_revol_bal"]].skew().round(2))

print("\nDimensions :", df_fe.shape)
print(df_fe.columns.tolist())

Asymétrie AVANT :
annual_inc    41.20
revol_bal     11.75
dtype: float64

Asymétrie APRÈS :
log_annual_inc    0.19
log_revol_bal    -2.75
dtype: float64

Dimensions : (395207, 21)
['loan_amnt', 'term', 'int_rate', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'dti', 'open_acc', 'pub_rec', 'revol_util', 'total_acc', 'initial_list_status', 'application_type', 'mort_acc', 'pub_rec_bankruptcies', 'annee_emission', 'anciennete_credit', 'ratio_pret_revenu', 'log_annual_inc', 'log_revol_bal']


In [28]:
cat_cols = df_fe.select_dtypes("object").columns.tolist()
print("Colonnes catégorielles :", cat_cols)

for col in cat_cols:
    print(f"\n--- {col} ---")
    print(pd.concat([df_fe[col].value_counts(),
                     (df_fe[col].value_counts(normalize=True) * 100).round(2)],
                    axis=1, keys=["nombre", "%"]))

Colonnes catégorielles : ['home_ownership', 'verification_status', 'purpose', 'initial_list_status', 'application_type']

--- home_ownership ---
                nombre      %
home_ownership               
MORTGAGE        198014  50.10
RENT            159393  40.33
OWN              37658   9.53
OTHER              110   0.03
NONE                29   0.01
ANY                  3   0.00

--- verification_status ---
                     nombre      %
verification_status               
Verified             139447  35.28
Source Verified      131296  33.22
Not Verified         124464  31.49

--- purpose ---
                    nombre      %
purpose                          
debt_consolidation  234161  59.25
credit_card          82922  20.98
home_improvement     23961   6.06
other                21056   5.33
major_purchase        8756   2.22
small_business        5656   1.43
car                   4670   1.18
medical               4175   1.06
moving                2842   0.72
vacation            

In [29]:
# 1) Regrouper les catégories rares
df_fe["home_ownership"] = df_fe["home_ownership"].replace({"NONE": "OTHER", "ANY": "OTHER"})
df_fe["purpose"] = df_fe["purpose"].replace({"renewable_energy": "other", "educational": "other"})

# 2) Supprimer la colonne presque constante
df_fe = df_fe.drop(columns=["application_type"])

print(df_fe["home_ownership"].value_counts())
print("\nNombre de catégories dans purpose :", df_fe["purpose"].nunique())

# 3) Encodage one-hot
cat_cols = ["home_ownership", "verification_status", "purpose", "initial_list_status"]
df_model = pd.get_dummies(df_fe, columns=cat_cols, drop_first=True, dtype=int)

print("\ndf_fe (lisible) :", df_fe.shape)
print("df_model (encodé) :", df_model.shape)
df_model.head(3)

home_ownership
MORTGAGE    198014
RENT        159393
OWN          37658
OTHER          142
Name: count, dtype: int64

Nombre de catégories dans purpose : 12

df_fe (lisible) : (395207, 20)
df_model (encodé) : (395207, 33)


,loan_amnt,term,int_rate,emp_length,dti,open_acc,pub_rec,revol_util,total_acc,mort_acc,pub_rec_bankruptcies,annee_emission,anciennete_credit,ratio_pret_revenu,log_annual_inc,log_revol_bal,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,verification_status_Source Verified,verification_status_Verified,purpose_credit_card,purpose_debt_consolidation,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_small_business,purpose_vacation,purpose_wedding,initial_list_status_w
0,10000.0,36,11.44,10.0,26.24,16.0,0.0,41.8,25.0,0.0,0.0,2015,24.6,0.085,11.669938,10.501500,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1
1,8000.0,36,11.99,4.0,22.05,17.0,0.0,53.3,27.0,3.0,0.0,2015,10.5,0.123,11.082158,9.910066,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,15600.0,36,10.49,0.0,12.79,13.0,0.0,92.2,26.0,0.0,0.0,2015,7.4,0.362,10.670303,9.391661,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0


In [30]:
print("Colonnes texte restantes :", df_model.select_dtypes("object").shape[1])
print("\nManquants restants :")
print(df_model.isnull().sum()[df_model.isnull().sum() > 0])

df_fe.to_parquet(chemin + "df_fe_phase1.parquet", index=False)
df_model.to_parquet(chemin + "df_model_phase1.parquet", index=False)
print("\nSauvegardé ✅", df_fe.shape, df_model.shape)

Colonnes texte restantes : 0

Manquants restants :
emp_length    18284
mort_acc      37205
dtype: int64

Sauvegardé ✅ (395207, 20) (395207, 33)
